### Oskar Goluda s490083

# Sprawozdanie projektu: Przewidywanie ryzyka Udaru za pomocą modelu TabNet

## Spis Treści:
### 1. Przedstawienie tematu
### 2. Cele projektu 
### 3. Opis problemu
### 4. Dane wejściowe i wyjściowe
### 5. Opis algorytmu Tabelarycznej Sieci Neuronowej - TabNet
### 6. Opis algorytmu Regresji Logarytmicznej
### 7. Wyniki i wnioski Modelu TabNet do przewidywania Udaru 
### 8. Porównanie zaprezentowanych Metod

## 1. Przedstawienie tematu

Udar mózgu jest jedną z głównych przyczyn niepełnosprawności i śmiertelności na świecie. Wczesne wykrywanie ryzyka wystąpienia udaru na podstawie parametrów zdrowotnych i stylu życia może kluczowo wpłynąć na prewencję i szybką interwencję medyczną.

Niniejszy projekt koncentruje się na zastosowaniu technik uczenia maszynowego do analizy danych medycznych i przewidywania prawdopodobieństwa wystąpienia udaru. Głównym narzędziem modelowania jest **TabNet** – nowoczesna architektura sieci neuronowej zaprojektowana specjalnie do wydajnej pracy z danymi tabelarycznymi. Dla celów porównawczych wykorzystamy również klasyczną **Regresję Logistyczną**, oraz porównamy wyniki.

### Zbiór danych
Projekt wykorzystuje zbiór danych `healthcare-dataset-stroke-data.csv` udostępniony na platformie kaggle, który zawiera informacje o pacjentach, takie jak:
*   **Dane demograficzne:** Płeć, wiek, stan cywilny, typ zamieszkania.
*   **Dane medyczne:** Nadciśnienie, choroby serca, poziom glukozy, BMI.
*   **Styl życia:** Rodzaj pracy, status palenia tytoniu.

Zbiór ten zostanie wykorzystany do wyuczenia sieci neuronowej przewidującej ryzyko wystąpienia Udaru w oparciu o paramety. Największą przeszkodą będzie zauważalne w tym zbiorze niezrównoważenie klas - niedobór przypadów klasy 1: udaru.

## 2. Cele projektu

Głównym celem projektu jest stworzenie skutecznego narzędzia wspomagającego diagnozę ryzyka udaru. Zrealizowano to poprzez następujące cele szczegółowe:

1.  **Analiza i przygotowanie danych (obsugiwane przez skrypty z pliku data_preparation.py):**
    *   Czyszczenie danych (obsługa brakujących wartości, np. BMI).
    *   Inżynieria cech (Feature Engineering): Stworzenie nowych zmiennych takich jak grupy wiekowe (`age_group`), kategorie BMI (`bmi_category`) czy sumaryczny licznik czynników zwiększonego ryzyka (`risk_factor_count`).
    *   Transformacja danych: Kodowanie zmiennych kategorycznych i skalowanie zmiennych numerycznych.

2.  **Modelowanie z użyciem TabNet (plik train.py):**
    *   Implementacja i trening modelu TabNet, który łączy zalety drzew decyzyjnych (interpretowalność) z mocą głębokich sieci neuronowych.
    *   Dobór odpowiednich hiperparametrów (np. `n_steps`, `width`, `lambda_sparse`).

3.  **Ewaluacja i Porównanie:**
    *   Ocena jakości modelu za pomocą metryk takich jak: Accuracy, Precision, Recall, F1-Score oraz ROC-AUC.
    *   Porównanie wyników TabNet z modelem referencyjnym (Regresją Logistyczną) w celu weryfikacji skuteczności bardziej złożonego podejścia.

4.  **Interpretowalność i Wizualizacja:**
    *   Analiza ważności cech (Feature Importance) w celu zrozumienia, które czynniki mają największy wpływ na ryzyko udaru.
    *   Wizualizacja wyników za pomocą macierzy pomyłek (Confusion Matrix) oraz krzywej ROC.

## 3. Opis problemu klasyfikacji

Definicja zadania
W ramach projektu rozwiązujemy problem binarnej klasyfikacji nadzorowanej. Zadaniem modelu jest przypisanie każdemu pacjentowi jednej z dwóch rozłącznych etykiet oznaczających wystąpienie udaru ($1$) lub jego brak ($0$) na podstawie zestawu parametrów medycznych i demograficznych.

Sformułowanie matematyczne
Niech $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^N$ będzie zbiorem danych treningowych składającym się z $N$ próbek, gdzie:
* $x_i \in \mathbb{R}^d$ to $d$-wymiarowy wektor cech (input) opisujący $i$-tego pacjenta.
* $y_i \in \{0, 1\}$ to rzeczywista etykieta klasy (target).

Celem procesu uczenia jest znalezienie funkcji (modelu) $f_\theta: \mathbb{R}^d \to [0, 1]$, sparametryzowanej przez wektor wag $\theta$, która dla danej próbki $x_i$ estymuje prawdopodobieństwo warunkowe przynależności do klasy pozytywnej:
$$ \hat{y}_i = f_\theta(x_i) = P(y_i = 1 | x_i) $$

Ostateczna predykcja klasy $\hat{c}_i$ odbywa się poprzez zastosowanie progu decyzyjnego $t$:
$$ \hat{c}_i = \begin{cases} 1 & \text{dla } \hat{y}_i \ge t \\ 0 & \text{dla } \hat{y}_i < t \end{cases} $$

Optymalizacja i oczekiwany wynik
Oczekiwanym wynikiem jest minimalizacja błędu predykcji na danych testowych.

Minimalizacja tej funkcji metodą spadku gradientu pozwala modelowi TabNet na naukę nieliniowych zależności w danych tabelarycznych przy jednoczesnym zachowaniu rzadkości wybieranych cech.

##   4. Dane wejściowe i wyjściowe

Źródło i format:
Dane pochodzą z platformy Kaggle i zostały umieszczone w  pliku healthcare-dataset-stroke-data.csv umieszczonego w katalogu data/raw/. Jest to plik w formacie CSV, zawierający rekordy medyczne pacjentów.

Struktura cech :
Zbiór danych zawiera następujące zmienne wejściowe (cechy):

| Nazwa cechy | Typ | Opis |
|:--- |:--- |:--- |
| **gender** | Słowna | Płeć pacjenta (Male, Female, Other). |
| **age** | Numeryczna | Wiek pacjenta. |
| **hypertension** | Binarna | 0: brak nadciśnienia, 1: stwierdzone nadciśnienie. |
| **heart_disease** | Binarna | 0: brak chorób serca, 1: stwierdzone choroby serca. |
| **ever_married** | Słowna | Czy pacjent kiedykolwiek był w małżeństwie (No, Yes). |
| **work_type** | Słowna | Rodzaj pracy (children, Govt_job, Private, Self-employed, Never_worked). |
| **Residence_type** | Słowna | Miejsce zamieszkania (Rural, Urban). |
| **avg_glucose_level** | Numeryczna | Średni poziom glukozy we krwi. |
| **bmi** | Numeryczna | Wskaźnik masy ciała (Body Mass Index). |
| **smoking_status** | Słowna | Status palenia (formerly smoked, never smoked, smokes, Unknown). |


Zmienna celu:
* stroke – Zmienna binarna przybierająca wartość 1, jeśli pacjent przeszedł udar, oraz 0, jeśli nie.

Podział danych (Data Split):
Dane zostały podzielone na trzy niezależne podzbiory w celu rzetelnej ewaluacji modelu Zastosowano podział warstwowy, aby zachować identyczne proporcje przypadków udaru (klasy 1) w każdym ze zbiorów, co jest kluczowe przy niezbalansowanych danych.
<br>
1. Zbiór treningowy (70%): Służy do trenowania modelu.
2. Zbiór walidacyjny (15%): Używany do monitorowania postępów uczenia i tuningu hiperparametrów (Early Stopping).
3. Zbiór testowy (15%): Wykorzystywany wyłącznie do ostatecznej oceny skuteczności modelu i stworzenia metryk oceniających jego działanie.

### Sposób przygotowania danych (Replikacja wyników)

Aby zreplikować wyniki eksperymentu, zaimplementowano w projekcie automatyczny pipeline do przetwarzania danych, który wykonuje następujące kroki:

1. Czyszczenie danych:
* Usunięcie kolumny identyfikacyjnej id.
* Usunięcie wierszy z brakującymi wartościami w kolumnie bmi.
* Usunięcie błędnych rekordów (np. wiek <= 0).
2. Inżynieria cech (Feature Engineering):
* Dyskretyzacja zmiennych ciągłych: utworzenie grup wiekowych (age_group), kategorii BMI (bmi_category) oraz poziomów glukozy.
* Agregacja ryzyka: stworzenie zmiennej risk_factor_count sumującej występowanie nadciśnienia, chorób serca i palenia tytoniu.
3. Transformacja:
* Label Encoding: Zamiana zmiennych kategorycznych (tekstowych) na wartości liczbowe.
* Standard Scaling: Standaryzacja zmiennych numerycznych (age, bmi, avg_glucose_level) do średniej 0 i odchylenia standardowego 1.

Instrukcja uruchomienia:

Aby pobrać dane, przetworzyć je i uruchomić trening modelu (co automatycznie wygeneruje pliki w folderze data/processed oraz wyniki w results/), należy wykonać z poziomu głównego katalogu projektu polecenie:
```bash
python3 src/train.py
```

### Dane wejsciowe do predykcji

Dane wejsciowe są zapisane w pliku `example_input.csv` w formacie CSV Plik ten służy do przekazywania nowych, nieprzetworzonych danych o pacjentach do skryptu inferencyjnego (src/infer.py).

Format pliku:
Każdy wiersz reprezentuje pojedynczy przypadek medyczny, a kolumny muszą odpowiadać oryginalnym nazwom cech ze zbioru treningowego. Przykładowa zawartość pliku wygląda następująco:

```csv
gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status
Female,54,1,0,Yes,Private,Urban,171.23,31.4,never smoked
Male,45,0,0,Yes,Self-employed,Rural,98.45,27.8,smokes
```

Po wykonaniu polecenia:
```bash
python3 src/infer.py --input example_input.csv
```
Dane te zostają przetworzone tak jak w przypadku danych ze zbiorów treningowego, walidacyjnego i testowego.Czyli są one poddane transformacji: Label Encoding'gowi, Scaling'owi oraz Feature Engineering'owi.

Tak przygotowany tensor przekazuwany jest do modelu TabNet w celu uzyskania predykcji,

### Dane wyjsciowe po predykcji

Po wykonaniu powyższego polecenia system generuje dane wyjściowe w dwóch formach:

1. Podsumowanie w terminalu.
Skrypt wyświetla w konsoli tabelaryczne zestawienie dla każdego pacjenta z pliku wejściowego.
Zawiera ono:
* Prawdopodobieństwo: Wyliczone przez model TabNet prawdopodobieństwo wystąpienia udaru (wartość od 0 do 1).
* Predykcja: Klasyfikacja binarna (1: Ryzyko udaru, 0: Brak ryzyka) na podstawie progu decyzyjnego.

2. Plik z wynikami `predictions.csv`:
W głównym katalogu projektu tworzony (lub nadpisywany) jest plik predictions.csv. Zawiera on kompletne wyniki, które można wykorzystać do dalszej analizy lub raportowania. 
Struktura pliku obejmuje:
* Wszystkie oryginalne kolumny wejściowe (dla łatwej identyfikacji pacjenta oraz możiwości dalszej analizy danych).
* Kolumnę stroke_probability – precyzyjny wynik pewności modelu.
* Kolumnę stroke_prediction – ostateczną decyzję klasyfikatora.


## 5. Opis algorytmu Tabelarycznej Sieci Neuronowej - TabNet

W projekcie zastosowano architekturę TabNet, opracowaną przez Google. Jest to zaawansowana architektura deep lerningowa dedykowana do danych tabelarycznych.

Zaimplementowany w pliku model (TabNetStrokeModel) opiera się na wielokrokowym przetwarzaniu danych wejściowych. Proces decyzyjny został skonfigurowany w następujący sposób (na podstawie 'src/train.py' i 'src/model.py'):

### Mechanizm Sekwencyjny
Model przetwarza dane w iteracyjnych krokach decyzyjnych ($N_{steps}$). <br>
<br>
Liczba kroków ($N_{steps}$): 4. Oznacza to, że sieć czterokrotnie "ogląda" dane, za każdym razem skupiając się na innym podzbiorze cech, agregując wiedzę z każdego kroku do finalnej predykcji.

### Transformator Cech

Każdy krok decyzyjny zawiera blok przetwarzania cech, który składa się z warstw:
1. Warstwy współdzielone ($N_{shared} = 2$): Neurony, których wagi są wspólne dla wszystkich kroków $N_{steps}$. Pozwala to na uczenie się globalnych wzorców w danych.
2. Warstwy niezależne ($N_{independent} = 2$): Neurony specyficzne dla danego kroku, pozwalające na specjalizację w wydobywaniu konkretnych detali.
3. GLU - Gated Linear Units zamiast standardowych funkcji aktywacji (jak ReLU), TabNet wykorzystuje GLU do kontrolowania przepływu informacji, co zapobiega problemowi zanikającego gradientu w głębokich strukturach.

$$ \text{GLU}(\mathbf{x}) = \sigma(\mathbf{x} \cdot \mathbf{W}_1 + \mathbf{b}_1) \odot
  (\mathbf{x} \cdot \mathbf{W}_2 + \mathbf{b}_2) $$

Gdzie:
* $\sigma$ – funkcja aktywacji sigmoidalna (decyduje o przepływie: 0-zamknięte, 1-otwarte).
* $\odot$ – iloczyn Hadamarda (mnożenie element po elemencie).
* $\mathbf{W}, \mathbf{b}$ – wyuczalne wagi i biasy.


Szerokość warstw decyzyjnych i atencji (width) została ustalona na 128 neuronów.

### Attentive Transformer i Maskowanie 
Kluczowym elementem algorytmu jest Attention Transformer, który decyduje, które cechy są brane pod uwagę w danym kroku.
* Wykorzystano funkcję aktywacji Sparsemax (zamiast Softmax). Sparsemax rzutuje wejścia na simplex prawdopodobieństwa, ale w przeciwieństwie do Softmax, potrafi przypisać dokładnie zerowe wagi nieistotnym cechom.
* W rezultacie powstaje maska cech (Sparse Mask), co sprawia, że model działa podobnie do "miękkiego" drzewa decyzyjnego, eliminując szum z danych (np. nieistotne wyniki badań dla konkretnego pacjenta).
* Zastosowano współczynnik relaksacji rzadkości (lambda_sparse = $1e-3$), który karze model za używanie zbyt wielu cech naraz, wymuszając skupienie się na najważniejszych predyktorach klinicznych (np. wiek, poziom glukozy).

  $$ \mathbf{M}[i] = \text{sparsemax}(\mathbf{P}[i-1] \cdot \mathbf{h}_i(\mathbf{a}[i-1])) $$

Gdzie:
* $\mathbf{M}[i]$ – maska cech w kroku $i$ (wskazuje, które cechy są ważne teraz).
* $\mathbf{P}[i-1]$ – "Prior scales" (informacja o tym, które cechy były użyte w poprzednich krokach).
* $\text{sparsemax}$ – funkcja, która (w przeciwieństwie do softmax) potrafi zerować wagi, tworząc rzadką (sparse) selekcję cech.


### Proces Uczenia i Training Loop

1. Funkcja Straty: Model minimalizuje błąd klasyfikacji oraz dodatkowy składnik kary za entropię maski.
2. Ghost Batch Normalization: W celu obsługi dużych wariancji w danych (typowych dla małych batchy), zastosowano wirtualny rozmiar wsadu (virtual_batch_size = 128) z parametrem momentum 0.02. Pozwala to na stabilniejszą estymację średniej i wariancji cech.
3. Optymalizacja: Użyto algorytmu z dynamicznym współczynnikiem uczenia (lr = 0.02) oraz mechanizmem Early Stopping. Trening jest przerywany, jeśli metryka walidacyjna (AUC - Area Under Curve) nie poprawia się przez 20 epok (patience=20), co zapobiega przeuczeniu przeuczeniu, czyli overfittinowi).

## 6. Opis algorytmu Regresji Logistycznej

Regresja Logistyczna została wykorzystana w projekcie jako model bazowy, służący do oceny, o ile bardziej złożona architektura TabNet poprawia wyniki predykcji. Jest to statystyczny model klasyfikacji binarnej, który oblicza prawdopodobieństwo przynależności próbkicdo danej klasy (udar / brak udaru).

Kluczowe cechy implementacji:
1. Funkcja Logistyczna (Sigmoid): Model mapuje kombinację liniową cech wejściowych na wartość z przedziału (0, 1) za pomocą wzoru:
      $$ P(y=1|x) = \frac{1}{1 + e^{-(w \cdot x + b)}} $$
2. Obsługa niezbalansowania klas: W pliku compare_logistic.py zastosowano parametr class_weight='balanced'. Jest to kluczowe, ponieważ w zbiorze danych przypadki udaru stanowią mniejszość – waga "balanced" automatycznie zwiększa karę za błędną klasyfikacj rzadkiej klasy (pacjentów z udarem), wymuszając na modelu większą czułość.
3. Liniowość: W przeciwieństwie do TabNetu, regresja logistyczna zakłada liniową zależność między logarytmem szansy (logit) a cechami, co czyni ją modelem prostym, szybkim, ale mniej skutecznym w wykrywaniu złożonych, nieliniowych korelacji medycznych.

## 7. Wyniki i wnioski Modelu TabNet do przewidywania Udaru 

Najlepsze wyniki jakie udało się uzyskać dostosowaując parametry metodą testowania empirycznego
Metryki dla zbioru testowego:
* accuracy: 0.7679,
* precision: 0.1153,
* recall: 0.6774,
* f1: 0.1971,
* roc_auc: 0.8239

Ważność cech:
<br>feature  importance
* age: 0.764497
* age_group:    0.085665
* avg_glucose_level:    0.061123
* risk_factor_count:    0.041928
* ever_married:    0.013593
* bmi_category:    0.010851
* work_type:    0.009539
* smoking_status:    0.006179
* Residence_type:    0.004585
* hypertension:    0.002033
* gender:    0.000008
* heart_disease:    0.000000
* bmi:    0.000000
* glucose_level_category:    0.000000

Macierz pomyłek:
| | Predicted 0 | Predicted 1 |
|---|---|---|
| **True 0** | 545 | 161 |
| **True 1** | 10 | 21 |


### Wnioski:
Na podstawie uzyskanych wyników można stwierdzić, że model TabNet częściowo radzi sobie z wykrywaniem przypadków udaru, jednak jego użyteczność predykcyjna w obecnej formie jest ograniczona przez kilka istotnych czynników: silną dysproporcję klas, niewielką liczbę próbek oraz, przez to, ze jest to model do wykrywania przypadkow medycznych, zbyt agresywne czułości kosztem precyzji. 

Zbiór danych zawiera ok. 5000 obserwacji, z czego jedynie 5% stanowią przypadki udaru. To jest mało jak na model głębokiego uczenia, nawet tak „lekki” jak TabNet. W praktyce oznacza to, że model uczy się głównie rozpoznawania klasy „brak udaru”, a klasa pozytywna jest reprezentowana bardzo słabo.

potwierdzają to wyniki z macierzy pomylek:tylko 31 rzeczywistych przypadków udaru w zbiorze testowym, 21 z nich wykrytych poprawnie, ale aż 161 fałszywych alarmów.

jest to celowy zabieg by model "strzelał" w przypadki udarów zbyt często, by nie przeoczyć prawdziwych przypadów.


### Interpretacja metryk:

1. Recall = 0.6774 - Model wykrywa większość przypadków udaru i jest to jedyna metryka z której jestem usatysfakcjonowany jak na wykorzystanie tak małego datasetu.
2. Precision = 0.1153 To jest bardzo niska wartość. Oznacza, że prawie 9 na 10 predykcji „udar” jest błędna. W praktyce klinicznej taki model generowałby masę niepotrzebnych alarmów i badań.
3. F1 = 0.1971 Niska wartość pokazuje, że balans między recall a precision jest słaby. Model jest wyraźnie przechylony w stronę „lepiej złapać za dużo niż za mało”.
4. Accuracy = 0.7679 - Metryka ta mało w przypadku nizrównowazenia cech w datasecie nie powinna być brana pod uwagę przy wyciąganiu wniąsków
5. ROC AUC = 0.8239 - AUC sugeruje, że model potrafi sensownie rozróżniać klasy, ale próg decyzyjny jest ustawiony by model bezpiecznie przewidywał ilość udarów



### Podsumowanie wniąsków

Wykorzystanie modelu TabNet do predykcji udarów jest możliwe, jednak przy tak małej ilości danych jest ono nieskuteczne i nie miałoby zastosowania we wczesnej diagnozie udarow. 

## 8. Porównanie zaprezentowanych Metod

### wyniki regresji logistycznej 
Uywając tych samych danych dla modelu prostej regresji logistycznej metryki dla takiego modelu wygladają następująco:

Wyniki Regresji Logistycznej:
* Accuracy:   0.7368
* Precision:  0.1137
* Recall:     0.7742
* F1-Score:   0.1983
* ROC-AUC:    0.8189

Dane z confusion matrix:
Poprawnie wykryte udary: 24
Przeoczone udary:        7
Fałszywe alarmy:         187


### Podsumowanie:
Porównując model TabNet z klasyczną regresją logistyczną jesteśmy wstanie wyciągnać wniąsek, iż złożoność modelu nie przełożyła się na istotną poprawę jakości predykcji w warunkach ograniczonego i silnie niezrównoważonego zbioru danych.

Oba modele osiągają bardzo zbliżone wartości metryk kluczowych dla problemów medycznych: recall, precision oraz F1-score. Regresja logistyczna uzyskała nawet wyższy recall (0.7742 vs 0.6774) kosztem większej liczby fałszywych alarmów (187 vs 161), podczas gdy TabNet oferuje nieznacznie lepszą precyzję (0.1153 vs 0.1137), co w praktyce jest różnicą kosmetyczną i bez znaczenia klinicznego.